# LeetCode #1138 — Alphabet Board Path: Đi trên bảng chữ cái

**Medium · Python 3 · String, Simulation, Greedy, Grid, BFS (cách trực giác)**  
[Đề LeetCode](https://leetcode.com/problems/alphabet-board-path/) · [Đối chiếu ví dụ](https://leetcode.ca/2019-01-11-1138-Alphabet-Board-Path/)

```python
class Solution:
    def alphabetBoardPath(self, target: str) -> str:
        ...
```

Điểm dễ sai nhất là chữ **`z`**: hàng cuối chỉ có một ô `(5,0)`. Notebook đi từ cách mô phỏng/BFS đến cách tạo đường đi ngắn nhất bằng tọa độ, có kiểm chứng mọi bước thật sự đi trong bảng.

## 1. Mục tiêu bài học

1. Hiểu hàng/cột 0-based, cách biểu diễn lưới bằng `list[str]`.
2. Dùng `ord` và `divmod` để biến chữ cái thành tọa độ.
3. Mô phỏng `U/D/L/R/!`, biết phân biệt di chuyển và chọn chữ.
4. Viết cách trực giác **BFS** tìm đường giữa hai ô.
5. Từ khoảng cách Manhattan suy ra đường đi ngắn không cần BFS.
6. Giải thích vì sao phải **đi lên và sang trái trước**, đặc biệt ở `z`.

## 2. Đề bài và ví dụ gốc

Bảng chữ cái:

```text
    c0 c1 c2 c3 c4
r0   a  b  c  d  e
r1   f  g  h  i  j
r2   k  l  m  n  o
r3   p  q  r  s  t
r4   u  v  w  x  y
r5   z
```

Bắt đầu ở `a=(0,0)`. Có thể đi `U` (lên), `D` (xuống), `L` (trái), `R` (phải); mọi bước phải tới **ô tồn tại**. Dấu `!` **chọn chữ ở ô hiện tại**. Hãy tạo một chuỗi lệnh chọn đúng `target` với **số bước di chuyển ít nhất**; có thể có nhiều kết quả đúng.

| Input | Một output hợp lệ |
|---|---|
| `target="leet"` | `"DDR!UURRR!!DDD!"` |
| `target="code"` | `"RR!DDRR!UUL!R!"` |

**Ràng buộc:** `1 <= len(target) <= 100`, chỉ gồm chữ tiếng Anh thường `a-z`.

In [1]:
BOARD = ["abcde", "fghij", "klmno", "pqrst", "uvwxy", "z"]
for r,row in enumerate(BOARD):
    print(f'r={r}: {row}')
assert BOARD[5] == 'z' and len(BOARD[5]) == 1

r=0: abcde
r=1: fghij
r=2: klmno
r=3: pqrst
r=4: uvwxy
r=5: z


## 3. Python căn bản: chỉ số `board[r][c]`

`board[2]` chọn hàng số 2 → chuỗi `"klmno"`. Sau đó `board[2][1]` lấy ký tự thứ 2 của hàng, tức `'l'`. Vì chỉ số bắt đầu bằng 0 nên `'a'` nằm ở `(0,0)`, `'y'` ở `(4,4)`.

Chỉ số hàng/cột không hợp lệ sẽ gây lỗi. **Không thể đi vào `(5,1)`** vì hàng 5 chỉ chứa `'z'` ở cột 0.

In [2]:
print('BOARD[2] =',BOARD[2])
print('BOARD[2][1] =',BOARD[2][1])
print('Tại (4,4):', BOARD[4][4])
assert BOARD[2][1]=='l' and BOARD[4][4]=='y'

BOARD[2] = klmno
BOARD[2][1] = l
Tại (4,4): y


## 4. Python căn bản: `ord`, `chr`, `divmod`

Vì chữ thường liên tiếp theo mã Unicode:

- `ord('a')` = 97; `ord('c') - ord('a')` = 2.
- Gọi `index=ord(ch)-ord('a')`, ta có `index=0..25`.
- `divmod(index,5)` trả `(index // 5, index % 5)`, tức `(row,col)`.
- `chr(ord('a')+index)` đi ngược từ số sang ký tự.

Ví dụ `'l'`: index 11 ⇒ `divmod(11,5)=(2,1)`; `'z'`: index 25 ⇒ `(5,0)`.

In [3]:
for ch in ['a','c','l','y','z']:
    idx=ord(ch)-ord('a')
    row,col=divmod(idx,5)
    print(f'{ch}: index={idx}, row={row}, col={col}; nhận lại={chr(97+idx)}')
assert divmod(ord('z')-ord('a'),5)==(5,0)

a: index=0, row=0, col=0; nhận lại=a
c: index=2, row=0, col=2; nhận lại=c
l: index=11, row=2, col=1; nhận lại=l
y: index=24, row=4, col=4; nhận lại=y
z: index=25, row=5, col=0; nhận lại=z


## 5. Đặt tên hàm rõ ràng cho từng công việc

**`pos(ch)`** trả vị trí chữ. **`is_cell(r,c)`** kiểm tra ô hợp lệ bằng độ dài của **hàng cụ thể** (không mặc định mọi hàng đều dài 5). Tách hai chức năng này sẽ làm cả cách BFS và bộ kiểm chứng ngắn, dễ đọc.

In [4]:
def pos(ch):
    return divmod(ord(ch) - ord('a'), 5)

def is_cell(r,c):
    return 0 <= r < len(BOARD) and 0 <= c < len(BOARD[r])

assert pos('z')==(5,0) and pos('y')==(4,4)
assert is_cell(5,0) and not is_cell(5,1) and not is_cell(-1,0)
print('Tọa độ và kiểm tra ô hợp lệ: PASS')

Tọa độ và kiểm tra ô hợp lệ: PASS


## 6. Ý nghĩa lệnh di chuyển và lệnh `!`

- `U`: `(r,c) → (r-1,c)`.
- `D`: `(r,c) → (r+1,c)`.
- `L`: `(r,c) → (r,c-1)`.
- `R`: `(r,c) → (r,c+1)`.
- `!`: chọn chữ hiện tại, *không di chuyển*.

Ví dụ đi từ `a=(0,0)` đến `c=(0,2)` cần `RR!`. Hai dấu `!` liền nhau có nghĩa chọn **cùng chữ hai lần**, cần thiết cho `target='aa'`.

In [5]:
DIRECTIONS = {'U':(-1,0),'D':(1,0),'L':(0,-1),'R':(0,1)}
r=c=0
for command in 'RR':
    dr,dc=DIRECTIONS[command]
    r,c=r+dr,c+dc
    print(command,'->', (r,c), BOARD[r][c])
assert (r,c)==pos('c')

R -> (0, 1) b
R -> (0, 2) c


## 7. Tạo bộ mô phỏng độc lập để kiểm chứng đường đi

Một chuỗi có thể có số bước rất ngắn nhưng **đi ra khỏi bảng**. Vì thế kiểm tra kết quả không chỉ là đếm lệnh; ta mô phỏng từng bước. Nếu chạm ô không tồn tại, ném `ValueError` với vị trí lỗi.

In [6]:
def simulate_path(path, verbose=False):
    r=c=0
    chosen=[]
    for i,command in enumerate(path):
        if command=='!':
            chosen.append(BOARD[r][c])
            if verbose: print(f'{i:2} ! chọn {BOARD[r][c]} tại {(r,c)}')
        else:
            if command not in DIRECTIONS:
                raise ValueError(f'Lệnh lạ: {command!r}')
            dr,dc=DIRECTIONS[command]
            nr,nc=r+dr,c+dc
            if not is_cell(nr,nc):
                raise ValueError(f'Bước {i}: {command} từ {(r,c)} tới ô không tồn tại {(nr,nc)}')
            r,c=nr,nc
            if verbose: print(f'{i:2} {command} tới {(r,c)} ({BOARD[r][c]})')
    return ''.join(chosen)

assert simulate_path('RR!')=='c'
assert simulate_path('!!')=='aa'
print('Bộ mô phỏng: PASS')

Bộ mô phỏng: PASS


## 8. Thử sai có chủ đích: tại sao `z` đặc biệt?

Từ `y=(4,4)` đến `z=(5,0)`, nếu đi **xuống trước** thì bước đầu tới `(5,4)` — ô *không tồn tại*. Cần đi `LLLL` sang cột 0 **trước**, rồi `D`.

Ngược lại từ `z=(5,0)` đến `y=(4,4)`, cần đi **lên trước** (`U`) rồi mới được đi sang phải (`RRRR`).

In [7]:
# Tới y từ a là hợp lệ; nhưng sau đó D ngay thì rơi ra khỏi bảng.
bad_path='RRRRDDDD!DLLLL!'
try:
    simulate_path(bad_path)
except ValueError as exc:
    print('Phát hiện lỗi đúng như dự đoán:',exc)
else:
    raise AssertionError('Bộ mô phỏng phải chặn việc đi xuống từ y')
assert simulate_path('RRRRDDDD!LLLLD!') == 'yz'
assert simulate_path('DDDDD!URRRR!') == 'zy'

Phát hiện lỗi đúng như dự đoán: Bước 9: D từ (4, 4) tới ô không tồn tại (5, 4)


## 9. Cách dễ nghĩ: BFS tìm đường ngắn giữa hai chữ

Ta xem 26 ô như **đỉnh đồ thị**; hai ô nối với nhau nếu ta có thể di chuyển giữa chúng bằng đúng 1 bước. **BFS** (Breadth-First Search, duyệt theo bề rộng) khám phá các ô theo thứ tự khoảng cách: đi 1 bước, 2 bước, 3 bước... Do đó lần đầu tới ô đích là đường ngắn nhất.

Dùng `deque` và `set` ghi nhớ đã thăm; mỗi trạng thái là `(row,col,path)`.

In [8]:
from collections import deque

def bfs_between(start,finish):
    if start==finish:
        return ''
    q=deque([(start,'')])
    seen={start}
    while q:
        (r,c),path=q.popleft()
        for command,(dr,dc) in DIRECTIONS.items():
            nxt=(r+dr,c+dc)
            if not is_cell(*nxt) or nxt in seen:
                continue
            if nxt==finish:
                return path+command
            seen.add(nxt)
            q.append((nxt,path+command))
    raise AssertionError('26 ô nằm trong cùng một vùng liên thông')

print('a -> l:',bfs_between(pos('a'),pos('l')))
print('y -> z:',bfs_between(pos('y'),pos('z')))
assert len(bfs_between(pos('y'),pos('z')))==5

a -> l: DDR
y -> z: LLLLD


## 10. Ghép nhiều đường BFS để chọn cả `target`

Sau khi tìm đường đến một chữ, thêm `!`, rồi lấy vị trí chữ đó làm điểm xuất phát tiếp theo. Đây là cách **chắc chắn đúng**, dùng để kiểm tra bản tối ưu.

In [9]:
def alphabet_path_bfs(target):
    cur=pos('a')
    pieces=[]
    for ch in target:
        nxt=pos(ch)
        pieces.append(bfs_between(cur,nxt))
        pieces.append('!')
        cur=nxt
    return ''.join(pieces)

for target in ['leet','code','za','yz','aa']:
    path=alphabet_path_bfs(target)
    print(target, '->',path)
    assert simulate_path(path)==target

leet -> DDR!UURRR!!DDD!
code -> RR!DDRR!UUL!R!
za -> DDDDD!UUUUU!
yz -> DDDDRRRR!LLLLD!
aa -> !!


## 11. Điểm cần tối ưu: bài này có tọa độ, không nhất thiết cần BFS

Hai ô `(r1,c1)` và `(r2,c2)` cách nhau ít nhất

\[D = |r_1-r_2|+|c_1-c_2|\]

bước đi lên/xuống/trái/phải. Đây là **khoảng cách Manhattan**: mỗi bước chỉ thay đổi đúng 1 tọa độ một đơn vị, nên muốn đổi tổng cộng từng ấy đơn vị thì ít nhất cần `D` bước.

Nếu ta xây được một đường **hợp lệ** dài đúng `D`, đường đó tối ưu và không cần tìm BFS.

In [10]:
def manhattan(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])

for a,b in [('a','l'),('y','z'),('z','e'),('c','c')]:
    d=manhattan(pos(a),pos(b))
    print(a,'->',b,'khoảng cách tối thiểu:',d)
    assert len(bfs_between(pos(a),pos(b)))==d

a -> l khoảng cách tối thiểu: 3
y -> z khoảng cách tối thiểu: 5
z -> e khoảng cách tối thiểu: 9
c -> c khoảng cách tối thiểu: 0


## 12. Bước đột phá: chọn một thứ tự di chuyển luôn hợp lệ

Ta di chuyển theo thứ tự cố định:

**1. `U` lên → 2. `L` sang trái → 3. `D` xuống → 4. `R` sang phải → 5. `!` chọn chữ.**

Vì sao thứ tự này xử lý được `z`?

- **Rời `z`:** nếu muốn sang phải thì phải lên khỏi hàng 5 trước; `U` đứng trước `R`.
- **Tới `z`:** phải về cột 0 trước khi xuống hàng 5; `L` đứng trước `D`.
- Những ô khác đều thuộc hình chữ nhật 5×5, nên dịch dọc/ngang trong đó luôn hợp lệ.

Thứ tự trên là **một quy tắc đủ**, không phải thứ tự duy nhất đúng. Nó dùng đúng `|Δr|+|Δc|` lệnh, nên đạt cận dưới Manhattan.

## 13. Cú pháp nhân chuỗi và `max(x,0)`

Trong Python `'U' * 3` tạo `'UUU'`. Nếu số lần đi lên là `r-tr`, chỉ nên viết `U` khi số đó dương: `max(r-tr,0)`. Nếu không đi lên, `max(...)` trả 0 và `'U'*0` là chuỗi rỗng.

Tương tự:

```python
U: max(r-tr,0)
L: max(c-tc,0)
D: max(tr-r,0)
R: max(tc-c,0)
```

In [11]:
print('U'*3+'L'*2+'!')
print('U'*max(4-5,0), '(rỗng khi không cần đi lên)')
assert 'D'*5+'!' == 'DDDDD!'

UUULL!
 (rỗng khi không cần đi lên)


## 14. Viết hàm di chuyển ngắn nhất giữa hai ô

Hàm dưới đây **không dùng BFS**. Nó chỉ tạo số lệnh dọc và ngang bắt buộc rồi nối theo thứ tự an toàn. Viết riêng hàm nhỏ trước khi ghép vào `target` để dễ test.

In [12]:
def safe_manhattan_moves(start,finish):
    r,c=start
    tr,tc=finish
    return ('U'*max(r-tr,0)
            + 'L'*max(c-tc,0)
            + 'D'*max(tr-r,0)
            + 'R'*max(tc-c,0))

assert safe_manhattan_moves(pos('y'),pos('z'))=='LLLLD'
assert safe_manhattan_moves(pos('z'),pos('y'))=='URRRR'
assert safe_manhattan_moves(pos('a'),pos('z'))=='DDDDD'
print('Các cặp liên quan tới z: PASS')

Các cặp liên quan tới z: PASS


## 15. Lời giải tối ưu O(k) thời gian

Khởi tạo tọa độ ở `a`. Với từng chữ `ch`, tính tọa độ đích; nối đoạn di chuyển hợp lệ rồi nối dấu `!`; cập nhật tọa độ hiện tại. `parts` là danh sách đoạn chuỗi; `''.join(parts)` ghép một lần, tránh cộng dồn chuỗi lớn nhiều lần.

In [13]:
def alphabet_path_greedy(target):
    here=pos('a')
    parts=[]
    for ch in target:
        there=pos(ch)
        parts.append(safe_manhattan_moves(here,there))
        parts.append('!')
        here=there
    return ''.join(parts)

for target in ['leet','code','za','yz','zy','aa']:
    result=alphabet_path_greedy(target)
    print(f'{target:>5} -> {result}')
    assert simulate_path(result)==target
assert alphabet_path_greedy('leet')=='DDR!UURRR!!DDD!'

 leet -> DDR!UURRR!!DDD!
 code -> RR!DDRR!UUL!R!
   za -> DDDDD!UUUUU!
   yz -> DDDDRRRR!LLLLD!
   zy -> DDDDD!URRRR!
   aa -> !!


## 16. Trace bằng tay với `target="zy"`

| Chữ cần chọn | Tọa độ hiện tại | Tọa độ đích | Lệnh di chuyển | Lệnh chọn |
|---|---|---|---|---|
| `z` | `a=(0,0)` | `(5,0)` | `DDDDD` | `!` |
| `y` | `z=(5,0)` | `(4,4)` | `URRRR` | `!` |

Ghép lại: `DDDDD!URRRR!`. Nếu từ `z` làm `RRRRU` thì bước `R` đầu tiên đi ra ngoài bảng.

In [14]:
def trace_alphabet(target):
    here=pos('a')
    parts=[]
    for i,ch in enumerate(target):
        there=pos(ch)
        move=safe_manhattan_moves(here,there)
        print(f'i={i}, ch={ch}, từ {here} tới {there}, move={move!r}, chọn !')
        parts.extend([move,'!'])
        here=there
    result=''.join(parts)
    print('Đường đi:',result)
    print('Mô phỏng chọn ra:',simulate_path(result))
    return result

assert trace_alphabet('zy')=='DDDDD!URRRR!'

i=0, ch=z, từ (0, 0) tới (5, 0), move='DDDDD', chọn !
i=1, ch=y, từ (5, 0) tới (4, 4), move='URRRR', chọn !
Đường đi: DDDDD!URRRR!
Mô phỏng chọn ra: zy


## 17. Code nộp LeetCode đúng chữ ký

Mọi hàm phụ trong notebook chỉ để dạy. Bản nộp dưới đây **tự chứa đầy đủ**: không dùng biến `BOARD`, `pos`, hay hàm mô phỏng bên ngoài.

In [15]:
class Solution:
    def alphabetBoardPath(self, target: str) -> str:
        r=c=0
        parts=[]
        for ch in target:
            tr,tc=divmod(ord(ch)-ord('a'),5)
            parts.append('U'*max(r-tr,0))
            parts.append('L'*max(c-tc,0))
            parts.append('D'*max(tr-r,0))
            parts.append('R'*max(tc-c,0))
            parts.append('!')
            r,c=tr,tc
        return ''.join(parts)

assert Solution().alphabetBoardPath('leet')=='DDR!UURRR!!DDD!'
assert simulate_path(Solution().alphabetBoardPath('code'))=='code'
print('LeetCode format: PASS')

LeetCode format: PASS


## 18. Vì sao đúng và vì sao đường đi ngắn nhất?

**Bất biến:** sau khi xử lý `target[:i]`, các lệnh đã tạo đi tới đúng ô của ký tự cuối cùng trong tiền tố đó và dấu `!` đã chọn chính xác `i` ký tự.

- Ban đầu, chưa chọn ký tự nào và ta đứng tại `a`.
- Với ký tự kế tiếp, `safe_manhattan_moves` di chuyển đến đúng tọa độ mà không đi vào ô không tồn tại (quy tắc `U,L,D,R` xử lý riêng hàng `z`). Sau đó `!` chọn đúng chữ, duy trì bất biến.
- Mọi đường đi giữa hai ô cần ít nhất `|Δr|+|Δc|` bước; đoạn vừa sinh dùng **đúng bằng số đó**, nên không thể rút ngắn. Các đoạn độc lập nối với nhau tại vị trí các chữ cần chọn, nên tổng tối ưu.

## 19. Độ phức tạp (định nghĩa rõ kích thước)

Gọi `k=len(target)`.

- Mỗi ký tự tạo nhiều nhất một số **hằng định** lệnh (trên bảng 26 ô); thời gian **O(k)**.
- Đầu ra có độ dài O(k), nên bộ nhớ lưu kết quả cũng **O(k)**; bộ nhớ phụ chỉ để giữ `r,c` là O(1) nếu không tính đầu ra.
- BFS phiên bản trực giác duyệt tối đa `26` ô và tối đa 4 cạnh/ô **cho mỗi chữ**, tức O(26k) = O(k) trên bảng cố định; nhưng logic nặng và ít trực tiếp hơn.

## 20. Kiểm thử mẫu, biên và chữ `z`

Bài cho phép **nhiều chuỗi lệnh đúng**. Vì vậy không nên so chuỗi với đáp án mẫu một cách máy móc; cần **mô phỏng và đếm bước** để kiểm tra tối ưu.

In [16]:
for target in ['leet','code','a','z','zz','az','za','yz','zy','zzz','yyyyz','azaza','abcde','zzzzza']:
    result=Solution().alphabetBoardPath(target)
    decoded=simulate_path(result)
    bfs_result=alphabet_path_bfs(target)
    assert decoded==target,(target,result,decoded)
    assert len(result)==len(bfs_result),(target,result,bfs_result)
    assert result.count('!')==len(target)
    print(f'{target:>7}: PASS; số lệnh={len(result)}')

   leet: PASS; số lệnh=15
   code: PASS; số lệnh=14
      a: PASS; số lệnh=1
      z: PASS; số lệnh=6
     zz: PASS; số lệnh=7
     az: PASS; số lệnh=7
     za: PASS; số lệnh=12
     yz: PASS; số lệnh=15
     zy: PASS; số lệnh=12
    zzz: PASS; số lệnh=8
  yyyyz: PASS; số lệnh=18
  azaza: PASS; số lệnh=25
  abcde: PASS; số lệnh=9
 zzzzza: PASS; số lệnh=16


## 21. Kiểm thử toàn bộ 26×26 cặp chữ

Có `26*26=676` cặp `(a,b)`. Mỗi cặp này **khác với việc đoán vài ví dụ**: ta kiểm tra mọi kiểu chuyển giữa hai ô, kể cả `z`. Dùng BFS như một oracle độc lập cho độ dài ngắn nhất.

In [17]:
import string
letters=string.ascii_lowercase
for a in letters:
    for b in letters:
        candidate=safe_manhattan_moves(pos(a),pos(b))
        assert len(candidate)==len(bfs_between(pos(a),pos(b)))
        # Để mô phỏng từ a, tạo đường từ vị trí xuất phát chung a ở (0,0).
        whole=safe_manhattan_moves(pos('a'),pos(a))+'!'+candidate+'!'
        assert simulate_path(whole)==a+b
print('676 cặp chữ: đường hợp lệ và ngắn nhất — PASS')

676 cặp chữ: đường hợp lệ và ngắn nhất — PASS


## 22. Test ngẫu nhiên: target dài tối đa 100 ký tự

Cả bản tối ưu và BFS phải tạo đường đi có **cùng số lệnh** (có thể khác thứ tự bước). Một `target` dài giúp phát hiện lỗi không cập nhật tọa độ ở cuối vòng lặp.

In [18]:
import random
random.seed(1138)
for i in range(100):
    target=''.join(random.choices(letters,k=random.randint(1,100)))
    greedy=Solution().alphabetBoardPath(target)
    oracle=alphabet_path_bfs(target)
    assert simulate_path(greedy)==target
    assert len(greedy)==len(oracle)
print('100 chuỗi ngẫu nhiên (độ dài 1..100): PASS')

100 chuỗi ngẫu nhiên (độ dài 1..100): PASS


## 23. Các lỗi phổ biến: code sai → nguyên nhân → cách sửa

1. **D luôn trước L:** đi từ `y` tới `z` sẽ tới `(5,4)` sai.
2. **R luôn trước U:** đi từ `z` sang chữ hàng trên sẽ tới `(5,1)` sai.
3. **Quên dấu `!` khi lặp chữ:** `target='aa'` cần `!!`.
4. **Quên cập nhật `r,c`:** chữ sau lại xuất phát từ chữ trước nữa.
5. **Dùng `index//5` và `%5` nhưng quên hàng `z` chỉ có 1 ô:** tọa độ có thể đúng nhưng lộ trình không hợp lệ.
6. **Chỉ test output bằng so sánh chuỗi cố định:** có thể bỏ nhầm một đường ngắn khác nhưng vẫn hợp lệ.

In [19]:
def wrong_vertical_before_horizontal(start, finish):
    r,c=start;tr,tc=finish
    return ('U'*max(r-tr,0)+'D'*max(tr-r,0)
            +'L'*max(c-tc,0)+'R'*max(tc-c,0))

bad=wrong_vertical_before_horizontal(pos('y'),pos('z'))
good=safe_manhattan_moves(pos('y'),pos('z'))
print('Sai y -> z:',bad, '| Đúng:',good)
try:
    simulate_path(safe_manhattan_moves(pos('a'),pos('y'))+'!'+bad+'!')
except ValueError:
    print('Bộ mô phỏng phát hiện đường sai: PASS')
else:
    raise AssertionError('Sai lộ trình từ y đến z phải bị từ chối')

Sai y -> z: DLLLL | Đúng: LLLLD
Bộ mô phỏng phát hiện đường sai: PASS


## 24. Mẹo nhận dạng pattern, biến thể

- **Lưới nhỏ có đường đi:** BFS là cách nghĩ an toàn đầu tiên.
- **Chỉ di chuyển 4 hướng, không có vật cản:** thử cận dưới Manhattan trước.
- **Bảng không hình chữ nhật:** kiểm tra chỗ khuyết khi đặt thứ tự các bước.
- **Nhiều đáp án đúng:** kiểm tra ràng buộc (đường hợp lệ, chọn đúng ký tự, đạt số bước tối thiểu), không chỉ so một output mẫu.

**Biến thể:** Nếu bảng có những ô bị khóa tùy ý, quy tắc `U,L,D,R` có thể không còn cho đường hợp lệ; hãy quay về BFS trên các ô đi được.

## 25. Bài tập tự luyện từ dễ đến khó

**Bài 1:** Tự tính `pos('m')`, `pos('z')`, `pos('e')` và vị trí `board[3][2]`.

**Bài 2:** Dự đoán `simulate_path('!RR!!')` cho ra chuỗi gì?

**Bài 3 — TODO:** viết `exercise_position(ch)` mà không gọi `pos`.

**Bài 4 — TODO:** viết `exercise_move(start,finish)` tạo đường ngắn nhất đúng cho mọi cặp chữ, **không** gọi `safe_manhattan_moves`.

**Bài 5 — TODO:** viết `exercise_alphabet_board_path(target)` rồi kiểm tra với `target='azaza'`.

Đáp án đặt ở mục sau; không gọi các hàm TODO trong phần Run All.

In [20]:
def exercise_position(ch):
    # TODO: ord, divmod
    pass

def exercise_move(start,finish):
    # TODO: đi U trước; L trước D; R sau U
    pass

def exercise_alphabet_board_path(target):
    # TODO: duyệt từng chữ; dùng exercise_position, exercise_move, nối !
    pass

# Tự bỏ comment sau khi hoàn thành:
# assert exercise_position('z') == (5,0)
# assert exercise_move((4,4),(5,0)) == 'LLLLD'
# assert simulate_path(exercise_alphabet_board_path('azaza')) == 'azaza'

## 26. Gợi ý 3 mức cho các bài

| Bài | Gợi ý tư duy | Gợi ý cú pháp | Gần-code |
|---|---|---|---|
| 1 | Chữ cái thứ 13 là `m` | `divmod(index,5)` | `m` có index 12 |
| 2 | `!` chọn không di chuyển | `RR` tới `c` | Hai `!` ở cuối cùng là hai lần chọn `c` |
| 3 | Đổi chữ thành số 0..25 | `ord(ch)-ord('a')` | `divmod(...,5)` |
| 4 | Tránh rơi khỏi `z` | Tạo `'U' * ...` trước | Thứ tự `U,L,D,R` |
| 5 | Mỗi chữ cần đúng một `!` | `parts.append(...)` | Cập nhật `here=there` sau mỗi chữ |

## 27. Đáp án tham khảo (chỉ mở sau khi làm)

**Bài 1:** `m=(2,2)`, `z=(5,0)`, `e=(0,4)`, `board[3][2]='r'`.

**Bài 2:** `!` tại `a` chọn `'a'`, `RR` đến `c`, hai `!` chọn `'cc'` ⇒ kết quả **`"acc"`**.

Các bài còn lại có hàm giải mẫu sau đây. Bạn nên tự viết trước rồi đối chiếu.

In [21]:
def answer_position(ch):
    return divmod(ord(ch)-ord('a'),5)

def answer_move(start,finish):
    r,c=start;tr,tc=finish
    return ('U'*max(r-tr,0)+'L'*max(c-tc,0)
            +'D'*max(tr-r,0)+'R'*max(tc-c,0))

def answer_path(target):
    here=(0,0)
    parts=[]
    for ch in target:
        there=answer_position(ch)
        parts.extend([answer_move(here,there),'!'])
        here=there
    return ''.join(parts)

assert simulate_path('!RR!!')=='acc'
assert answer_position('m')==(2,2)
assert answer_move((4,4),(5,0))=='LLLLD'
assert simulate_path(answer_path('azaza'))=='azaza'
print('Đáp án tự luyện: PASS')

Đáp án tự luyện: PASS


## 28. Template và checklist submit

```text
Bắt đầu r=c=0 (a)
Với mỗi chữ ch:
  tr,tc = divmod(ord(ch)-ord('a'),5)
  U lên trước
  L sang trái
  D xuống (sau L)
  R sang phải (sau U)
  ! chọn chữ
  r,c = tr,tc
Trả chuỗi lệnh đã ghép
```

- [ ] Thử `z`, `az`, `za`, `yz`, `zy` và chữ lặp (`aa`, `zz`).
- [ ] Không bước nào đi ra khỏi `BOARD`.
- [ ] Có đúng `len(target)` dấu `!`.
- [ ] Số bước di chuyển đạt tổng khoảng cách Manhattan từng cặp chữ.
- [ ] Code nộp tự chứa, không cần các hàm test của notebook.